# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [6]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [7]:
# Task 1 — Multi-series line with highlight
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/co2_emissions.csv')
asia_df = df[df['Region'] == 'Asia']

fig = go.Figure()

highlight_country = 'China'
brand_gold = '#E3AC17'
brand_purple = '#1E0554'
grey_lines = '#DDDDDD'

for country in asia_df['Country'].unique():
    country_data = asia_df[asia_df['Country'] == country].sort_values('Year')
    
    is_target = (country == highlight_country)
    
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['CO2_Mt'],
        mode='lines',
        line=dict(
            color=brand_gold if is_target else grey_lines,
            width=3 if is_target else 1
        ),
        name=country,
        hoverinfo='name+y',
        showlegend=False
    ))

last_point = asia_df[asia_df['Country'] == highlight_country].iloc[-1]
fig.add_annotation(
    x=last_point['Year'],
    y=last_point['CO2_Mt'],
    text=f"  {highlight_country}",
    showarrow=False,
    xanchor='left',
    yanchor='middle',
    font=dict(color=brand_gold, size=14, family="Arial Black")
)

fig.update_layout(
    title=dict(
        text="<b>China's Exponential Growth</b>",
        font=dict(size=22, color=brand_purple),
        xanchor='center',
        x=0.5
    ),
    plot_bgcolor='white',
    xaxis=dict(title="Year", showgrid=False, linecolor=brand_purple),
    yaxis=dict(title="CO2 Emissions (Mt)", gridcolor='#F0F0F0', linecolor=brand_purple),
    margin=dict(r=100)
)

fig.show()

---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [8]:
# Task 2 — Slopegraph: regional averages
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/co2_emissions.csv')
years = [2000, 2022]
slope_df = df[df['Year'].isin(years)].groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()
pivot_df = slope_df.pivot(index='Region', columns='Year', values='CO2_Mt').dropna()
pivot_df.columns = ['y2000', 'y2022']

fig = go.Figure()

brand_purple = '#1E0554' 
brand_gold = '#E3AC17'    

pivot_df = pivot_df.sort_values('y2022', ascending=False)

for region in pivot_df.index:
    v0, v22 = pivot_df.loc[region, 'y2000'], pivot_df.loc[region, 'y2022']
    color = brand_gold if v22 > v0 else brand_purple
    
    fig.add_trace(go.Scatter(
        x=[2000, 2022], y=[v0, v22],
        mode='lines+markers',
        line=dict(color=color, width=4),
        marker=dict(color=color, size=10),
        hoverinfo='name+y'
    ))
    
    fig.add_annotation(
        x=2000, y=v0, text=f"{region} {v0:.1f} ",
        showarrow=False, xanchor='right',
        font=dict(color=color, size=11)
    )
    
    fig.add_annotation(
        x=2022, y=v22, text=f" {v22:.1f}",
        showarrow=False, xanchor='left',
        font=dict(color=color, size=11, family="Arial Black")
    )

fig.update_layout(
    title=dict(
        text="<b>Regional Divergence</b>",
        font=dict(size=20, color=brand_purple),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        showgrid=False, showline=False,
        tickvals=[2000, 2022], 
        range=[1990, 2030]
    ),
    yaxis=dict(
        showgrid=False, showticklabels=False,
        range=[-100, 4000]
    ),
    plot_bgcolor='white',
    showlegend=False,
    height=700,
    margin=dict(l=150, r=150)
)

fig.show()